## Part 1: Dataset
A Dataset is just a class that answers two questions:

How many samples do I have? → __len__
Give me sample number i → __getitem__

```python
from torch.utils.data import Dataset

class MyDataset(Dataset):
    def __init__(self, X, y):
        self.X = X          # Store features
        self.y = y          # Store labels
    
    def __len__(self):
        return len(self.X)  # Total Number of samples
    
    def __getitem__(self, i):
        return self.X[i], self.y[i]     # Return one sample by index
```

## Part 2: DataLoader
Once you have a Dataset, you wrap it in a DataLoader:

```python
from torch.utils.data import DataLoader

loader = DataLoader(
    dataset,        # Your Dataset object
    batch_size=32,  # How many samples per batch
    shuffle=True    # Shuffle every epoch (important for training!)
)
```

```python
for X_batch, y_batch in loader:
    # X_batch shape: (32, num_features)
    # y_batch shape: (32,)
    ...
```
The DataLoader handles all the batching and shuffling automatically.

# Full Working Example

In [2]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(42)

### 1. Create Dataset

In [3]:
class MyDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, i):
        return self.X[i], self.y[i]

### 2. Generate some data

In [4]:
X = torch.randn(100, 4)          # 100 samples, 4 features
y = torch.randint(0, 2, (100,))  # Binary labels

In [5]:
# Train/val split
X_train, y_train = X[:80], y[:80]
X_val,   y_val   = X[80:], y[80:]

### 3. Wrap in Dataset

In [6]:
train_dataset = MyDataset(X_train, y_train)
val_dataset   = MyDataset(X_val,   y_val)

### 4. Wrap in DataLoader

In [8]:
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=16, shuffle=False)
#                                                        ^^^^^^^^^^^^
#                                   No need to shuffle validation data

### 5. Simple model

In [9]:
model = nn.Sequential(
    nn.Linear(4, 32),
    nn.ReLU(),
    nn.Linear(32, 2)
)

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

### 6. Training loop with DataLoader

In [ ]:
for epoch in range(5):
    model.train()
    total_train_loss = 0

    for X_batch, y_batch in train_loader:   # DataLoader gives batches automatically
        optimizer.zero_grad()
        out = model(X_batch)
        loss = criterion(out, y_batch)
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()      # Accumulate loss across batches

    # Validation
    model.eval()    
    total_val_loss = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            out = model(X_batch)
            total_val_loss += criterion(out, y_batch).item()

    # Average loss = total loss / number of batches
    avg_train = total_train_loss / len(train_loader)
    avg_val   = total_val_loss   / len(val_loader)

    print(f"Epoch {epoch+1} | Train Loss: {avg_train:.4f} | Val Loss: {avg_val:.4f}")

Epoch 1 | Train Loss: 0.6927 | Val Loss: 0.7579
Epoch 2 | Train Loss: 0.6606 | Val Loss: 0.7136
Epoch 3 | Train Loss: 0.6452 | Val Loss: 0.6856
Epoch 4 | Train Loss: 0.6344 | Val Loss: 0.6708
Epoch 5 | Train Loss: 0.6255 | Val Loss: 0.6711
